# Clasificador de Tipo de Cuerpo

## Setup

In [11]:
!pip install torch torchvision onnx onnxruntime pillow numpy scikit-learn tqdm onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 19.8 MB/s eta 0:00:00


In [2]:
import os, torch, torch.nn as nn
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights
from PIL import Image
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import StratifiedShuffleSplit
from tqdm import tqdm
import random

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

CLASS_NAMES = ["ectomorfo", "endomorfo", "mesomorfo"]
NUM_CLASSES = len(CLASS_NAMES)
print(f"Classes: {CLASS_NAMES}")

Device: cuda
Classes: ['ectomorfo', 'endomorfo', 'mesomorfo']


## 1. Subida del Dataset para Colab

In [ ]:
from google.colab import files
import zipfile, io

print("Upload body_dataset.zip (recommended):")
uploaded = files.upload()
for fn in uploaded:
    with zipfile.ZipFile(io.BytesIO(uploaded[fn]), "r") as z:
        z.extractall("/content")
    print(f"Extracted: {fn}")

DATA_DIR = "/content/body_dataset"
for cls in CLASS_NAMES:
    path = f"{DATA_DIR}/{cls}"
    imgs = [f for f in os.listdir(path) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
    print(f"  {cls}: {len(imgs)} images")
print("Total images:", sum(len([f for f in os.listdir(f"{DATA_DIR}/{cls}") if f.lower().endswith((".jpg", ".jpeg", ".png"))]) for cls in CLASS_NAMES))

Upload body_dataset.zip (recommended):


Saving body_dataset.zip to body_dataset.zip
Extracted: body_dataset.zip
  ectomorfo: 100 images
  endomorfo: 106 images
  mesomorfo: 100 images
Total images: 306


## 2. Dataset Loader

In [ ]:
class BodyDataset(Dataset):
    def __init__(self, paths, labels, transform):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        img = self.transform(img)
        return img, self.labels[idx]


all_paths, all_labels = [], []
for cls_idx, cls_name in enumerate(CLASS_NAMES):
    cls_dir = "%s/%s" % (DATA_DIR, cls_name)
    for fn in sorted(os.listdir(cls_dir)):
        if fn.lower().endswith((".jpg", ".jpeg", ".png")):
            all_paths.append(os.path.join(cls_dir, fn))
            all_labels.append(cls_idx)

all_paths = np.array(all_paths)
all_labels = np.array(all_labels)
print("Total images: %d" % len(all_paths))
print("Class distribution: %s" % np.bincount(all_labels))

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, temp_idx = next(sss.split(all_paths, all_labels))

paths_train, labels_train = all_paths[train_idx], all_labels[train_idx]
paths_temp, labels_temp = all_paths[temp_idx], all_labels[temp_idx]

sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
val_idx, test_idx = next(sss2.split(paths_temp, labels_temp))
paths_val, labels_val = paths_temp[val_idx], labels_temp[val_idx]
paths_test, labels_test = paths_temp[test_idx], labels_temp[test_idx]

print("Train: %d | Val: %d | Test: %d" % (len(paths_train), len(paths_val), len(paths_test)))

train_transform = T.Compose([
    T.Resize((256, 256)),
    T.RandomResizedCrop(224, scale=(0.7, 1.0)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(20),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15, hue=0.05),
    T.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    T.RandomErasing(p=0.2, scale=(0.02, 0.1)),
])

val_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

ds_train = BodyDataset(paths_train, labels_train, train_transform)
ds_val   = BodyDataset(paths_val,   labels_val,   val_transform)
ds_test  = BodyDataset(paths_test,  labels_test,  val_transform)

dl_train = DataLoader(ds_train, batch_size=16, shuffle=True,  num_workers=2)
dl_val   = DataLoader(ds_val,   batch_size=16, shuffle=False, num_workers=2)
dl_test  = DataLoader(ds_test,  batch_size=16, shuffle=False, num_workers=2)

print("DataLoaders ready")


Total images: 306
Class distribution: [100 106 100]
Train: 244 | Val: 31 | Test: 31
DataLoaders ready


## 3. Modelo Pre-Entrenado: MobileNetV3 Small

In [ ]:
model = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1)

for param in model.features.parameters():
    param.requires_grad = False
for i in range(-2, 0):
    for param in model.features[i].parameters():
        param.requires_grad = True

in_features = model.classifier[3].in_features
model.classifier[3] = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(in_features, NUM_CLASSES),
)

model = model.to(DEVICE)

from sklearn.utils.class_weight import compute_class_weight
cw = compute_class_weight('balanced', classes=np.unique(labels_train), y=labels_train)
class_weights = torch.FloatTensor(cw).to(DEVICE)
print("Class weights: %s" % cw)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW([
    {'params': model.features[-2].parameters(), 'lr': 1e-5},
    {'params': model.features[-1].parameters(), 'lr': 1e-5},
    {'params': model.classifier.parameters(), 'lr': 1e-3},
], lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen = sum(p.numel() for p in model.parameters() if not p.requires_grad)
print("Trainable: %dK | Frozen: %.1fM" % (trainable/1000, frozen/1e6))
print("Output classes: %d" % NUM_CLASSES)


Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 202MB/s]


Class weights: [1.01666667 0.96825397 1.01666667]
Trainable: 944K | Frozen: 0.6M
Output classes: 3


## 4. Entrenamiento

In [ ]:
EPOCHS = 30
best_acc = 0.0

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(dl_train, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    scheduler.step()
    train_acc = 100.0 * correct / total

    model.eval()
    val_correct = 0
    val_total = 0
    val_loss = 0.0
    with torch.no_grad():
        for images, labels in dl_val:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()

    val_acc = 100.0 * val_correct / val_total
    print(f"Loss: {train_loss/len(dl_train):.4f}/{val_loss/len(dl_val):.4f} | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "/content/best_body_type.pth")
        print(f"  --> Saved (best val acc: {val_acc:.2f}%)")

print(f"\nBest validation accuracy: {best_acc:.2f}%")

# Test evaluation
model.load_state_dict(torch.load("/content/best_body_type.pth"))
model.eval()
test_correct = 0
test_total = 0
all_preds, all_true = [], []
with torch.no_grad():
    for images, labels in dl_test:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model(images)
        _, predicted = outputs.max(1)
        test_total += labels.size(0)
        test_correct += predicted.eq(labels).sum().item()
        all_preds.extend(predicted.cpu().numpy())
        all_true.extend(labels.cpu().numpy())

print(f"\nTest Accuracy: {100.0 * test_correct / test_total:.2f}%")
print("\nClassification Report:")
print(classification_report(all_true, all_preds, target_names=CLASS_NAMES, digits=3))

Epoch 1/30: 100%|██████████| 16/16 [00:03<00:00,  4.09it/s]


Loss: 1.1444/1.2118 | Train Acc: 40.16% | Val Acc: 29.03%
  --> Saved (best val acc: 29.03%)


Epoch 2/30: 100%|██████████| 16/16 [00:04<00:00,  3.92it/s]


Loss: 0.9116/1.6245 | Train Acc: 56.56% | Val Acc: 32.26%
  --> Saved (best val acc: 32.26%)


Epoch 3/30: 100%|██████████| 16/16 [00:02<00:00,  5.62it/s]


...

Loss: 0.4868/0.9366 | Train Acc: 81.15% | Val Acc: 64.52%


Epoch 29/30: 100%|██████████| 16/16 [00:02<00:00,  5.49it/s]


Loss: 0.5840/0.9277 | Train Acc: 75.41% | Val Acc: 64.52%


Epoch 30/30: 100%|██████████| 16/16 [00:02<00:00,  5.72it/s]


Loss: 0.5415/0.9211 | Train Acc: 78.28% | Val Acc: 67.74%

Best validation accuracy: 70.97%

Test Accuracy: 35.48%

Classification Report:
              precision    recall  f1-score   support

   ectomorfo      0.400     0.400     0.400        10
   endomorfo      0.000     0.000     0.000        11
   mesomorfo      0.438     0.700     0.538        10

    accuracy                          0.355        31
   macro avg      0.279     0.367     0.313        31
weighted avg      0.270     0.355     0.303        31



## 5. Exportar

In [12]:
import onnx
import os

model.load_state_dict(torch.load("/content/best_body_type.pth"))
model.eval()

dummy_input = torch.randn(1, 3, 224, 224).to(DEVICE)

# Export to temp file first (may create external .data)
torch.onnx.export(
    model,
    dummy_input,
    "/content/_body_temp.onnx",
    input_names=["input"],
    output_names=["output"],
    opset_version=17,
    dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},
)

# Reload and save as self-contained (no external .data file)
onnx_model = onnx.load("/content/_body_temp.onnx")
onnx.save_model(onnx_model, "/content/body_type_model.onnx", save_as_external_data=False)
os.remove("/content/_body_temp.onnx")

size = os.path.getsize("/content/body_type_model.onnx") / 1024 / 1024
print(f"Exported: /content/body_type_model.onnx ({size:.1f} MB)")

/tmp/ipykernel_1558/276209183.py:6: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0702 15:31:20.579000 1558 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `MobileNetV3([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `MobileNetV3([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_str, target_version)
                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: /project/onnx/version_converter/adapters/axes_input_to_attribute.h:56: adapt: Assertion `node-

[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
ONNX exported to /content/body_type_model.onnx
Size: 0.3 MB


## 6. Prueba

In [13]:
import onnxruntime as ort

session = ort.InferenceSession("/content/body_type_model.onnx")
input_name = session.get_inputs()[0].name

def predict(img_path):
    img = Image.open(img_path).convert("RGB")
    img = val_transform(img).unsqueeze(0).numpy()
    outputs = session.run(None, {input_name: img})[0]
    probs = torch.nn.functional.softmax(torch.from_numpy(outputs[0]), dim=0).numpy()
    pred_idx = np.argmax(probs)
    return CLASS_NAMES[pred_idx], float(probs[pred_idx])

# Test on one sample per class
for cls in CLASS_NAMES:
    sample = [f for f in os.listdir(f"{DATA_DIR}/{cls}") if f.lower().endswith((".jpg", ".jpeg", ".png"))][0]
    pred, conf = predict(f"{DATA_DIR}/{cls}/{sample}")
    print(f"Expected: {cls:12s} | Predicted: {pred:12s} | Confidence: {conf:.3f}")

Expected: ectomorfo    | Predicted: ectomorfo    | Confidence: 0.767
Expected: endomorfo    | Predicted: endomorfo    | Confidence: 0.537
Expected: mesomorfo    | Predicted: mesomorfo    | Confidence: 0.748


## 7. Descarga del modelo

In [14]:
from google.colab import files
files.download("/content/body_type_model.onnx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>